<a href="https://colab.research.google.com/github/KhantCode96/CCCD_identity_card_extraction/blob/main/PrepareDataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd "/content/drive/MyDrive/YOLOV8-VIET_OCR"

/content/drive/MyDrive/YOLOV8-VIET_OCR


In [ ]:
!pip install -r requirements.txt

In [ ]:
from google.colab import userdata
import os

# Access the ROBOFLOW_API_KEY from Colab's secrets manager
# This line will now correctly retrieve the API key if it's set up as a secret
os.environ["ROBOFLOW_API_KEY"] = userdata.get('ROBOFLOW_API_KEY')

# Now retry the Roboflow initialization
from roboflow import Roboflow

rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("ocrtestingdataset").project("cccd_dataset-xz2sc")
version = project.version(8)
dataset = version.download("yolov8")
print(dataset.location)

loading Roboflow workspace...
loading Roboflow project...
/content/drive/MyDrive/YOLOV8-VIET_OCR/CCCD_dataset-8


### Change label from cordinate into a box

In [ ]:
from pathlib import Path
import shutil

DATASET_ROOT = Path(
    "/content/drive/MyDrive/YOLOV8-VIET_OCR/CCCD_dataset-8"
)

SPLITS = ("train", "valid", "val", "test")


def convert_row(row: str, file_path: Path, line_number: int):
    parts = row.split()

    # Already a normal YOLO detection row:
    # class x_center y_center width height
    if len(parts) == 5:
        try:
            class_id = int(float(parts[0]))
            values = [float(value) for value in parts[1:]]
        except ValueError as exc:
            raise ValueError(
                f"{file_path}:{line_number}: non-numeric detection row: {row}"
            ) from exc

        if not all(0.0 <= value <= 1.0 for value in values):
            raise ValueError(
                f"{file_path}:{line_number}: coordinates outside 0..1: {row}"
            )

        return row, False

    # Polygon or oriented bounding box:
    # class x1 y1 x2 y2 ... xn yn
    coordinate_count = len(parts) - 1

    if coordinate_count < 6 or coordinate_count % 2 != 0:
        raise ValueError(
            f"{file_path}:{line_number}: unsupported row with "
            f"{len(parts)} values: {row}"
        )

    try:
        class_id = int(float(parts[0]))
        coordinates = [float(value) for value in parts[1:]]
    except ValueError as exc:
        raise ValueError(
            f"{file_path}:{line_number}: non-numeric polygon row: {row}"
        ) from exc

    if not all(0.0 <= value <= 1.0 for value in coordinates):
        raise ValueError(
            f"{file_path}:{line_number}: coordinates outside 0..1: {row}"
        )

    xs = coordinates[0::2]
    ys = coordinates[1::2]

    x_min = min(xs)
    x_max = max(xs)
    y_min = min(ys)
    y_max = max(ys)

    width = x_max - x_min
    height = y_max - y_min

    if width <= 0 or height <= 0:
        raise ValueError(
            f"{file_path}:{line_number}: zero-area annotation: {row}"
        )

    x_center = (x_min + x_max) / 2
    y_center = (y_min + y_max) / 2

    converted = (
        f"{class_id} "
        f"{x_center:.10f} "
        f"{y_center:.10f} "
        f"{width:.10f} "
        f"{height:.10f}"
    )

    return converted, True


label_directories = [
    DATASET_ROOT / split / "labels"
    for split in SPLITS
    if (DATASET_ROOT / split / "labels").exists()
]

if not label_directories:
    raise FileNotFoundError(
        f"No train/valid/val/test label directories found under {DATASET_ROOT}"
    )

# First pass: validate and prepare all converted contents.
# Nothing is modified if an invalid row is encountered.
planned_changes = {}
converted_rows = 0
total_rows = 0

for label_directory in label_directories:
    for label_file in sorted(label_directory.glob("*.txt")):
        output_rows = []

        for line_number, raw_row in enumerate(
            label_file.read_text(encoding="utf-8").splitlines(),
            start=1,
        ):
            row = raw_row.strip()

            if not row:
                continue

            converted_row, was_converted = convert_row(
                row,
                label_file,
                line_number,
            )

            output_rows.append(converted_row)
            total_rows += 1
            converted_rows += int(was_converted)

        content = "\n".join(output_rows)
        if output_rows:
            content += "\n"

        planned_changes[label_file] = content

# Back up every original labels directory.
for label_directory in label_directories:
    backup_directory = label_directory.with_name(
        "labels_polygon_backup"
    )

    if not backup_directory.exists():
        shutil.copytree(label_directory, backup_directory)
        print(f"Backup created: {backup_directory}")
    else:
        print(f"Backup already exists: {backup_directory}")

# Write converted labels.
for label_file, content in planned_changes.items():
    label_file.write_text(content, encoding="utf-8")

print()
print(f"Label files processed: {len(planned_changes)}")
print(f"Total annotation rows: {total_rows}")
print(f"Polygon/OBB rows converted: {converted_rows}")
print("Conversion completed.")

Backup created: /content/drive/MyDrive/YOLOV8-VIET_OCR/CCCD_dataset-8/train/labels_polygon_backup
Backup created: /content/drive/MyDrive/YOLOV8-VIET_OCR/CCCD_dataset-8/valid/labels_polygon_backup
Backup created: /content/drive/MyDrive/YOLOV8-VIET_OCR/CCCD_dataset-8/test/labels_polygon_backup

Label files processed: 3273
Total annotation rows: 21176
Polygon/OBB rows converted: 114
Conversion completed.


#### check if the label is valid after change

In [ ]:
from pathlib import Path

dataset_root = Path(
    "/content/drive/MyDrive/YOLOV8-VIET_OCR/CCCD_dataset-8"
)

invalid_rows = []

for split in ("train", "valid", "val", "test"):
    labels_dir = dataset_root / split / "labels"

    if not labels_dir.exists():
        continue

    for label_file in labels_dir.glob("*.txt"):
        for line_number, row in enumerate(
            label_file.read_text().splitlines(),
            start=1,
        ):
            if row.strip() and len(row.split()) != 5:
                invalid_rows.append(
                    (str(label_file), line_number, row)
                )

print("Remaining invalid rows:", len(invalid_rows))

for item in invalid_rows[:10]:
    print(item)

Remaining invalid rows: 0


### check if valid

In [ ]:
%%bash
cd /content/drive/MyDrive/YOLOV8-VIET_OCR/CCCD_dataset-8  # Change directory to the dataset root
ls -lah /content/drive/MyDrive/YOLOV8-VIET_OCR/tools/train_yolov8_cccd_colab.py # Check script path, now relative to new CWD
ls -lah data.yaml # Check data.yaml path, now relative to new CWD
python /content/drive/MyDrive/YOLOV8-VIET_OCR/tools/validate_yolov8_dataset.py --data-yaml data.yaml # Run validation, adjust script and data.yaml paths

-rw------- 1 root root 6.7K Jul 14 02:03 /content/drive/MyDrive/YOLOV8-VIET_OCR/tools/train_yolov8_cccd_colab.py
-rw------- 1 root root 407 Jul 14 08:25 data.yaml
Classes: ['birth', 'bottom_left', 'bottom_right', 'id', 'name', 'top_left', 'top_right']
train: images=2660 labels=17275
  birth: 2539
  bottom_left: 2413
  bottom_right: 2416
  id: 2543
  name: 2548
  top_left: 2419
  top_right: 2397
val: images=550 labels=3493
  birth: 528
  bottom_left: 488
  bottom_right: 486
  id: 526
  name: 526
  top_left: 473
  top_right: 466
test: images=63 labels=408
  birth: 62
  bottom_left: 56
  bottom_right: 57
  id: 62
  name: 62
  top_left: 55
  top_right: 54
Dataset validation passed.
